In [20]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_csv(r"C:\Users\Shivni\Desktop\Marketing_analysis\data\raw\tech_advertising_campaigns_dataset.csv")

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   campaign_id                   10000 non-null  object 
 1   campaign_objective            10000 non-null  object 
 2   platform                      10000 non-null  object 
 3   ad_placement                  10000 non-null  object 
 4   device_type                   10000 non-null  object 
 5   operating_system              10000 non-null  object 
 6   creative_format               10000 non-null  object 
 7   creative_size                 10000 non-null  object 
 8   ad_copy_length                10000 non-null  object 
 9   has_call_to_action            10000 non-null  bool   
 10  creative_emotion              10000 non-null  object 
 11  creative_age_days             10000 non-null  int64  
 12  target_audience_age           10000 non-null  object 
 13  ta

In [23]:
df["start_date"] = pd.to_datetime(df["start_date"])

In [24]:
df["start_date"].dtype

dtype('<M8[ns]')

In [25]:
df["start_date"].dt.year

0       2024
1       2024
2       2025
3       2024
4       2025
        ... 
9995    2026
9996    2025
9997    2024
9998    2025
9999    2024
Name: start_date, Length: 10000, dtype: int32

In [26]:
df["start_date"].dt.month

0        3
1        1
2        5
3        7
4        3
        ..
9995     1
9996     1
9997    10
9998    12
9999     8
Name: start_date, Length: 10000, dtype: int32

In [27]:
df.shape

(10000, 41)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   campaign_id                   10000 non-null  object        
 1   campaign_objective            10000 non-null  object        
 2   platform                      10000 non-null  object        
 3   ad_placement                  10000 non-null  object        
 4   device_type                   10000 non-null  object        
 5   operating_system              10000 non-null  object        
 6   creative_format               10000 non-null  object        
 7   creative_size                 10000 non-null  object        
 8   ad_copy_length                10000 non-null  object        
 9   has_call_to_action            10000 non-null  bool          
 10  creative_emotion              10000 non-null  object        
 11  creative_age_days            

In [29]:
print("Earliest Campaign:",df["start_date"].min())
print("Latest Campaign:", df["start_date"].max())

Earliest Campaign: 2024-01-01 00:00:00
Latest Campaign: 2026-01-30 00:00:00


In [30]:
df["quarter_calc"] = df["start_date"].dt.quarter
(df["quarter_calc"] != df["quarter"]).sum()

np.int64(0)

In [31]:
df.drop(columns = ["quarter_calc"], inplace = True)

In [32]:
df.shape

(10000, 41)

In [33]:
text_cols = df.select_dtypes(include = ["object"]).columns

for col in text_cols:
    leading_spaces = df[col].astype(str).str.startswith(" ").sum()
    trailing_spaces = df[col].astype(str).str.endswith(" ").sum()

    if leading_spaces > 0 or trailing_spaces > 0:
        print(f"{col}: leading = {leading_space}, trailing = {trailing_space}")

In [34]:
df["CPA_clean"] = df["ad_spend"] / df["conversions"].replace(0, pd.NA)

In [35]:
df["CPA_clean"].isna().sum()

np.int64(594)

In [36]:
df["conversion_status"] = df["conversions"].apply(
    lambda x: "Converted" if x>0 else "No Conversion")

In [37]:
df["conversion_status"].value_counts()

conversion_status
Converted        9406
No Conversion     594
Name: count, dtype: int64

In [38]:
df[["ad_spend", "conversions", "CPA_clean", "conversion_status"]].head(10)

,ad_spend,conversions,CPA_clean,conversion_status
0,896.46,2,448.23,Converted
1,1158.16,5,231.632,Converted
2,434.56,6,72.426667,Converted
3,1019.67,3,339.89,Converted
4,425.82,0,<NA>,No Conversion
5,2411.98,4,602.995,Converted
6,11544.20,85,135.814118,Converted
7,1438.40,4,359.6,Converted
8,1153.28,9,128.142222,Converted
9,4187.79,100,41.8779,Converted


In [39]:
df.loc[df["conversions"]>0, "CPA_clean"].describe()

count     9406.0
unique    9284.0
top        136.8
freq         4.0
Name: CPA_clean, dtype: float64

In [40]:
print("Infinite CPA values:", np.isinf(df["CPA_clean"].dropna()).sum())

TypeError: ufunc 'isinf' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [41]:
df["CPA_clean"].dtype

dtype('O')

In [42]:
df["CPA_clean"] = pd.to_numeric(df["CPA_clean"], errors = "coerce")

In [43]:
df["CPA_clean"].dtype

dtype('float64')

In [44]:
print("Infinite CPA values:", np.isinf(df["CPA_clean"].dropna()).sum())

Infinite CPA values: 0


In [45]:
print("Missing CPA values:", df["CPA_clean"].isna().sum())

Missing CPA values: 594


In [46]:
df.to_csv(r"C:\Users\Shivni\Desktop\Marketing_analysis\data\cleaned\marketing_campaigns_cleaned.csv")